<a href="https://colab.research.google.com/github/Goat-Seller/bankruptcy-prediction/blob/main/data_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# dane
data = pd.read_csv("data.csv")
data.columns = data.columns.str.strip()

target = "Bankrupt?"
x = data.drop(columns=[target, "Net Income Flag"])
y = data[target]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=1, stratify=y
)

chosen_features = [
    'Persistent EPS in the Last Four Seasons',
    'Total debt/Total net worth',
    'Non-industry income and expenditure/revenue',
    'Net Value Growth Rate',
    'Quick Ratio',
    'Tax rate (A)',
    'Cash/Total Assets',
    'Operating Gross Margin',
]

scaler = StandardScaler()
x_train_s_chosen = scaler.fit_transform(x_train[chosen_features])
x_test_s_chosen = scaler.transform(x_test[chosen_features])




In [6]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

def oversample(X, y, seed=0):
    rng = np.random.RandomState(seed)
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    k = len(neg) // len(pos)
    idx = np.concatenate([neg, np.tile(pos, k)])
    rng.shuffle(idx)
    return X[idx], y[idx]

def augment(X, y, sd=0.1, seed=0):
    rng = np.random.RandomState(seed)
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    k = len(neg) // len(pos) - 1
    X_new = np.tile(X[pos], (k, 1))
    X_new = X_new + rng.normal(0, sd, X_new.shape)
    X_all = np.vstack([X, X_new])
    y_all = np.concatenate([y, np.ones(len(X_new), dtype=int)])
    idx = rng.permutation(len(y_all))
    return X_all[idx], y_all[idx]

y_tr = np.array(y_train)
y_te = np.array(y_test)

variants = {
    "plain": (x_train_s_chosen, y_tr),
    "oversampling": oversample(x_train_s_chosen, y_tr),
    "augmentation sd=0.1": augment(x_train_s_chosen, y_tr, sd=0.1),
    "augmentation sd=0.3": augment(x_train_s_chosen, y_tr, sd=0.3),
}

results = []
for name, (X, y) in variants.items():
    clf = MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=500, random_state=42)
    clf.fit(X, y)
    pred = clf.predict(x_test_s_chosen)
    results.append({
        "variant": name,
        "accuracy": accuracy_score(y_te, pred),
        "recall": recall_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "F1": f1_score(y_te, pred),
    })

display(pd.DataFrame(results).round(3))

,variant,accuracy,recall,precision,F1
0,plain,0.970,0.114,0.714,0.196
1,oversampling,0.908,0.523,0.180,0.267
2,augmentation sd=0.1,0.964,0.136,0.353,0.197
3,augmentation sd=0.3,0.968,0.182,0.500,0.267


Data augmentation affects the results of MLP drastically. Event with augmentation it performs worse than Linear Regression or SVM so it will not be developed. Recall is too low